# MVSR – Vorbereitung Einheit 2
## Entrauschen, Kernels und Segmentierung

**Bearbeitungszeit:** ca. 30–45 Minuten

Dieses Notebook bereitet die Inhalte der zweiten MVSR-Einheit praktisch vor. Im Mittelpunkt stehen **klassische Bildverarbeitungsmethoden**, mit denen aus Rohbildern relevante Informationen für robotische Anwendungen gewonnen werden können.

Sie untersuchen dazu lokale Bildoperationen mit Kernels, verschiedene Möglichkeiten zum Glätten und Entrauschen sowie eine einfache Segmentierung über Pixelwerte und Schwellenwerte.

Für dieses Binder-Notebook wird **C++17 mit OpenCV 4.6** verwendet.

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- erklären, warum Rohbilder vor der weiteren Verarbeitung häufig aufbereitet werden,
- beschreiben, wie ein Kernel lokale Pixelnachbarschaften verarbeitet,
- einfache Glättungs-, Schärfungs- und Kantenfilter mit OpenCV anwenden,
- eine binäre Maske mit Schwellenwerten erzeugen,
- RGB/BGR- und HSV-basierte Segmentierung vergleichen,
- zusammenhängende Regionen über Konturen und Bounding Boxes beschreiben,
- Grenzen einer klassischen, regelbasierten Segmentierung einschätzen.

> **Hinweis:** Ziel ist nicht, bereits alle Verfahren vollständig zu beherrschen. Das Notebook soll die grundlegenden Ideen vor der Vorlesung praktisch erfahrbar machen.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook wird über **MyBinder** direkt im Browser ausgeführt. Der vorhandene C++-Code kann unmittelbar in den Notebook-Zellen bearbeitet und ausgeführt werden; eine zusätzliche lokale Installation ist dafür nicht erforderlich.

Das Notebook besteht aus Text- und Codezellen. Die Textzellen enthalten kurze Erklärungen und Aufgabenstellungen, die C++-Codezellen können direkt ausgeführt und verändert werden.

Eine Codezelle wird über den **Play-Button** oder mit **Shift + Enter** ausgeführt. Die Ausgabe erscheint anschließend direkt unterhalb der jeweiligen Zelle.

Arbeiten Sie das Notebook am besten **von oben nach unten** durch, da spätere Codezellen teilweise auf zuvor definierten Variablen und Funktionen aufbauen.

Bei Aufgaben mit `TODO` sollen Sie den vorhandenen Code selbstständig ergänzen oder verändern.

C++ wird mit **xeus-cling** inkrementell ausgeführt. Dadurch können bereits deklarierte Variablen bei einer erneuten Ausführung derselben Zelle zu einer Fehlermeldung führen. Wo dies für eine Übungszelle relevant ist, werden lokale Blöcke oder Lambda-Funktionen verwendet. Falls nötig, starten Sie den Kernel neu und führen Sie die Zellen erneut von oben nach unten aus.

> **Wichtig:** Änderungen innerhalb einer Binder-Sitzung werden nicht dauerhaft im GitHub-Repository gespeichert.

## 0. Setup

Die Binder-Umgebung ist bereits mit **xeus-cling** und **OpenCV 4.6** vorbereitet.

> **Binder-/Notebook-Hinweis:** Die folgenden `#pragma cling`-Anweisungen sind ein Workaround für den interaktiven C++-Kernel. Sie teilen `xeus-cling` mit, wo die OpenCV-Header und die kompilierten Bibliotheken liegen. In einem normalen lokalen C++-Projekt wird OpenCV stattdessen beim Kompilieren bzw. über das Build-System, z.B. mit CMake, eingebunden und gelinkt. Im eigentlichen C++-Quellcode genügt dann üblicherweise `#include <opencv2/opencv.hpp>`.

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>
#include <fstream>

// OpenCV-Pfade für Binder / Ubuntu
#pragma cling add_include_path("/usr/include/opencv4")
#pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

// Benötigte OpenCV-Bibliotheken laden
#pragma cling load("opencv_core")
#pragma cling load("opencv_imgproc")
#pragma cling load("opencv_imgcodecs")
#pragma cling load("opencv_features2d")
#pragma cling load("opencv_calib3d")

#include <opencv2/opencv.hpp>

std::cout << "OpenCV-Version: " << CV_VERSION << std::endl;
std::cout << "C++ Standard: " << __cplusplus << std::endl;

### Hilfsfunktion zur Darstellung

OpenCV verwendet in einem lokalen C++-Programm normalerweise `cv::imshow()` zur Bilddarstellung. Dabei wird ein eigenes Fenster geöffnet, z.B.

```cpp
cv::imshow("Bild", image);
cv::waitKey(0);
```

In Binder läuft das Notebook jedoch im Browser und besitzt keine normale Desktop-GUI.

> **Binder-/Notebook-Hinweis:** Die folgende Hilfsfunktion ist daher ein Workaround für die Browser-Umgebung. Sie kodiert eine `cv::Mat` als PNG und übergibt sie an die Rich-Display-Funktion des Jupyter-Kernels. Damit ein Bild dargestellt wird, muss `show_image(...)` als **letzte Expression einer Zelle ohne Semikolon** stehen.

In [ ]:
#include "nlohmann/json.hpp"
#include "xtl/xbase64.hpp"

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(const cv::Mat& image)
        {
            std::vector<unsigned char> buffer;
            cv::imencode(".png", image, buffer);

            png_data.assign(
                reinterpret_cast<const char*>(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(const NotebookImage& image)
    {
        auto bundle = nl::json::object();
        bundle["image/png"] = xtl::base64encode(image.png_data);
        return bundle;
    }
}

mvsr::NotebookImage show_image(const cv::Mat& image)
{
    return mvsr::NotebookImage(image);
}

## 1. Vom Rohbild zu relevanten Informationen

Ein Kamerabild enthält zunächst nur **Pixelwerte**. Für die Robotik interessieren uns aber häufig bestimmte Strukturen, z.B.

- gleichmäßige Regionen,
- bestimmte Farben,
- Kanten und Linien,
- zusammenhängende Bereiche eines Objekts.

In realen Kamerabildern kommen zusätzlich Störungen hinzu, etwa Sensorrauschen, Quantisierung, Bewegungsunschärfe oder Kompressionsartefakte.

Für die folgenden Versuche verwenden wir das FHTW Logo. Sie können gerne das Bild gegen eines der anderen Testbilder austauschen oder ein eigenes Bild in ihren Colab workspace abspeichern und verwenden.

* ***Für andere Testbilder müssen die Parameter entsprechend angepasst werden!***
* Die weiteren Testbilder sind im folgendem Codeblock auskommentiert

> **Binder-/Notebook-Hinweis:** MyBinder klont beim Start das vollständige GitHub-Repository. Das Testbild kann daher direkt aus dem Ordner `test_images` des Repositories geladen werden. Da das Notebook selbst in einem Unterordner liegen kann, werden unten mehrere mögliche relative Pfade geprüft. In einem normalen lokalen C++-Projekt würde man bei bekanntem Dateipfad einfach `cv::imread("pfad/zum/bild.png")` verwenden.

In [ ]:
cv::Mat image;
std::string image_path;

/*std::vector<std::string> possible_paths = {
    "test_images/fhtw_logo_low_res.png",
    "../test_images/fhtw_logo_low_res.png",
    "../../test_images/fhtw_logo_low_res.png",
    "../../../test_images/fhtw_logo_low_res.png"
};*/

std::vector<std::string> possible_paths = {
    "test_images/fhtw_logo.png",
    "../test_images/fhtw_logo.png",
    "../../test_images/fhtw_logo.png",
    "../../../test_images/fhtw_logo.png"
};


for (const auto& path : possible_paths)
{
    if (std::ifstream(path).good())
    {
        image = cv::imread(path, cv::IMREAD_COLOR);
        image_path = path;
        break;
    }
}

if (image.empty())
{
    std::cerr << "Testbild konnte nicht gefunden werden." << std::endl;
}
else
{
    std::cout << "Geladen: " << image_path << std::endl;
    std::cout << "Bildauflösung (H, W, C): " << image.rows << "," << image.cols << "," << image.channels() << std::endl;
}

> **Binder-/Notebook-Hinweis:** Für die Darstellung wird im Notebook `show_image(...)` verwendet. In einem lokalen C++-Programm würde das Bild normalerweise mit `cv::imshow()` und `cv::waitKey()` angezeigt.

In [ ]:
show_image(image)

### Rauschen hinzufügen

Ein idealer Sensor würde für jeden Pixel exakt den gewünschten Messwert liefern. Reale Sensorwerte schwanken jedoch. Um diesen Effekt zu untersuchen, fügen wir der Szene künstliches gaußförmiges Rauschen hinzu.

Der Parameter `noise_sigma` bestimmt die Stärke des Rauschens.

In [ ]:
double noise_sigma = 25.0;

cv::Mat image_float;
image.convertTo(image_float, CV_32FC3);

cv::Mat noise(image.size(), CV_32FC3);

cv::RNG rng(42);
rng.fill(
    noise,
    cv::RNG::NORMAL,
    cv::Scalar::all(0.0),
    cv::Scalar::all(noise_sigma)
);

cv::Mat image_noisy_float = image_float + noise;

cv::Mat image_noisy;
image_noisy_float.convertTo(image_noisy, CV_8UC3);

> **Binder-/Notebook-Hinweis:** In der Python-/Colab-Version können mehrere Bilder komfortabel in einer gemeinsamen Matplotlib-Abbildung dargestellt werden. Die Rich-Display-Lösung in diesem C++-Notebook zeigt dagegen jeweils das Ergebnis der letzten Expression einer Zelle. Deshalb werden Vergleichsbilder in getrennten Zellen dargestellt. Lokal könnten mehrere `cv::imshow()`-Fenster gleichzeitig verwendet werden.

### Original

In [ ]:
show_image(image)

### Verrauschtes Bild

In [ ]:
show_image(image_noisy)

### Probieren Sie selbst

Verändern Sie die Stärke des Rauschens.

- Ab welchem Wert werden homogene Flächen deutlich unruhig?
- Was passiert an Kanten?
- Welche Bildbereiche erscheinen besonders störanfällig?

> **Binder-/Notebook-Hinweis:** Die Schreibweise `[](){ ... }()` erzeugt eine kleine anonyme Funktion (Lambda), die direkt ausgeführt wird. Sie wird hier verwendet, damit die Variablen der Übungszelle lokal bleiben und die Zelle nach Änderungen leichter erneut ausgeführt werden kann. In einem normalen lokalen C++-Programm wäre diese zusätzliche Lambda-Konstruktion dafür nicht notwendig.

In [ ]:
[]()
{
    // TODO: Rauschstärke verändern
    double noise_sigma_test = 100.0;

    cv::Mat image_float_test;
    image.convertTo(image_float_test, CV_32FC3);

    cv::Mat noise_test(image.size(), CV_32FC3);
    cv::RNG rng_test(42);

    rng_test.fill(
        noise_test,
        cv::RNG::NORMAL,
        cv::Scalar::all(0.0),
        cv::Scalar::all(noise_sigma_test)
    );

    cv::Mat image_noisy_test_float = image_float_test + noise_test;

    cv::Mat image_noisy_test;
    image_noisy_test_float.convertTo(image_noisy_test, CV_8UC3);

    std::cout << "noise_sigma = " << noise_sigma_test << std::endl;

    return show_image(image_noisy_test);
}()

### Optionaler Exkurs: Kompression

Bilder werden häufig nicht als rohe Pixelarrays gespeichert. JPEG komprimiert verlustbehaftet und kann dadurch insbesondere an Kanten sichtbare Artefakte erzeugen.

Für klassische Bildverarbeitung ist das relevant, weil Segmentierung und Kantendetektion direkt auf den Pixelwerten arbeiten.

Die folgende Zelle kodiert dasselbe Bild mit niedriger JPEG-Qualität und dekodiert es wieder.

In [ ]:
int jpeg_quality = 50;

std::vector<int> encode_params = {
    cv::IMWRITE_JPEG_QUALITY,
    jpeg_quality
};

std::vector<unsigned char> encoded_jpg;

bool success = cv::imencode(
    ".jpg",
    image,
    encoded_jpg,
    encode_params
);

cv::Mat image_jpeg = cv::imdecode(
    encoded_jpg,
    cv::IMREAD_COLOR
);

cv::Mat difference;
cv::absdiff(image, image_jpeg, difference);

std::cout << "JPEG-Dateigröße: "
          << encoded_jpg.size()
          << " Byte" << std::endl;

> **Binder-/Notebook-Hinweis:** Die drei Ergebnisse werden wieder getrennt dargestellt. Lokal könnte man dafür drei `cv::imshow()`-Fenster öffnen.

### Original

In [ ]:
show_image(image)

### JPEG Qualität = 50

In [ ]:
show_image(image_jpeg)

### Absolute Differenz

In [ ]:
show_image(difference)

## 2. Lokale Nachbarschaften und Kernels

Ein einzelner Pixel ist nur ein Messwert. Seine **Nachbarschaft** enthält jedoch zusätzliche Struktur.

Ein Kernel ist eine kleine Gewichtsmatrix, die über das Bild bewegt wird. An jeder Position werden die Pixelwerte der lokalen Nachbarschaft gewichtet und aufsummiert.

Für einen Kernel \(K\) kann die lokale Filteroperation vereinfacht als

$g(x,y)=\sum_i\sum_j K(i,j)\,I(x+i,y+j)$

geschrieben werden.

Je nach Wahl der Gewichte kann ein Kernel beispielsweise

- glätten,
- schärfen,
- oder lokale Übergänge hervorheben.

### Eine lokale gewichtete Summe

Wir betrachten zunächst nur eine kleine $3\times3$-Nachbarschaft. Ein Box-Filter gewichtet alle neun Pixel gleich stark.

In [ ]:
cv::Mat patch = (cv::Mat_<float>(3, 3) <<
    100, 110, 120,
    105, 200, 115,
    100, 108, 112
);

cv::Mat box_kernel = cv::Mat::ones(3, 3, CV_32F) / 9.0f;

double filtered_value = cv::sum(
    patch.mul(box_kernel)
)[0];

std::cout << "Bildnachbarschaft:" << std::endl;
std::cout << patch << std::endl;

std::cout << "\nKernel:" << std::endl;
std::cout << box_kernel << std::endl;

std::cout << "\nGefilterter Wert im Zentrum: "
          << filtered_value << std::endl;

Der auffällige Zentralwert `200` wird durch die Mittelung mit seinen Nachbarn abgeschwächt. Genau diese Idee wird beim Glätten auf das gesamte Bild angewendet.

OpenCV stellt dafür sowohl fertige Filterfunktionen als auch `cv::filter2D()` für selbst definierte Kernels bereit.

### Typische Kernels

Wir vergleichen vier einfache Filter:

**Identität**

$\begin{bmatrix}0&0&0\\0&1&0\\0&0&0\end{bmatrix}$

**Box Blur**

$\frac{1}{9}\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}$

**Gaussian Blur**

$\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}$

**Schärfen**

$\begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}$

In [ ]:
cv::Mat kernel_identity = (cv::Mat_<float>(3, 3) <<
    0, 0, 0,
    0, 1, 0,
    0, 0, 0
);

cv::Mat kernel_box = cv::Mat::ones(3, 3, CV_32F) / 9.0f;

cv::Mat kernel_gaussian = (cv::Mat_<float>(3, 3) <<
    1, 2, 1,
    2, 4, 2,
    1, 2, 1
) / 16.0f;

cv::Mat kernel_sharpen = (cv::Mat_<float>(3, 3) <<
     0, -1,  0,
    -1,  5, -1,
     0, -1,  0
);

cv::Mat image_identity;
cv::Mat image_box;
cv::Mat image_gaussian_kernel;
cv::Mat image_sharpen;

cv::filter2D(
    image_noisy,
    image_identity,
    -1,
    kernel_identity
);

cv::filter2D(
    image_noisy,
    image_box,
    -1,
    kernel_box
);

cv::filter2D(
    image_noisy,
    image_gaussian_kernel,
    -1,
    kernel_gaussian
);

cv::filter2D(
    image,
    image_sharpen,
    -1,
    kernel_sharpen
);

> **Binder-/Notebook-Hinweis:** Wie zuvor werden die Filterergebnisse in getrennten Zellen angezeigt. Lokal könnten mehrere `cv::imshow()`-Aufrufe direkt nacheinander verwendet werden.

### Identität

In [ ]:
show_image(image_identity)

### Box Blur

In [ ]:
show_image(image_box)

### Gaussian Kernel

In [ ]:
show_image(image_gaussian_kernel)

### Schärfen

In [ ]:
show_image(image_sharpen)

### Probieren Sie selbst

Verändern Sie den Schärfungs-Kernel oder definieren Sie einen eigenen $3\times3$-Kernel.

Beobachten Sie:

- Welche Strukturen werden verstärkt?
- Entstehen neue Artefakte?
- Was passiert, wenn die Summe aller Kernelgewichte deutlich größer als 1 wird?

> **Binder-/Notebook-Hinweis:** Die Übung wird wieder in einer unmittelbar ausgeführten Lambda-Funktion gekapselt. Dadurch können lokale Variablen beim Experimentieren erneut erzeugt werden. In einem normalen lokalen C++-Programm wäre diese zusätzliche Kapselung nicht erforderlich.

In [ ]:
[]()
{
    // TODO: Eigenen Kernel definieren
    cv::Mat kernel_test = (cv::Mat_<float>(3, 3) <<
         0, -1,  0,
        -1,  5, -1,
         0, -1,  0
    );

    cv::Mat image_kernel_test;

    cv::filter2D(
        image,
        image_kernel_test,
        -1,
        kernel_test
    );

    std::cout << "Summe der Kernelgewichte: "
              << cv::sum(kernel_test)[0]
              << std::endl;

    return show_image(image_kernel_test);
}()

## 3. Glätten und Entrauschen

Beim Entrauschen soll eine Störung reduziert werden, ohne die für die weitere Verarbeitung wichtigen Strukturen vollständig zu verlieren.

Zwei einfache Möglichkeiten sind:

- **Box Blur:** alle Pixel der Nachbarschaft werden gleich gewichtet,
- **Gaussian Blur:** Pixel nahe am Zentrum werden stärker gewichtet.

Größere Filterfenster glätten stärker, verwischen aber auch Kanten stärker.

In [ ]:
cv::Mat box_blur;
cv::Mat gaussian_blur;

cv::blur(
    image_noisy,
    box_blur,
    cv::Size(5, 5)
);

cv::GaussianBlur(
    image_noisy,
    gaussian_blur,
    cv::Size(5, 5),
    0
);

> **Binder-/Notebook-Hinweis:** Die drei Bilder werden für den Vergleich getrennt ausgegeben. Lokal könnten sie mit mehreren `cv::imshow()`-Fenstern gleichzeitig dargestellt werden.

### Verrauscht

In [ ]:
show_image(image_noisy)

### Box Blur 5x5

In [ ]:
show_image(box_blur)

### Gaussian Blur 5x5

In [ ]:
show_image(gaussian_blur)

### Mini-Aufgabe: Filtergröße untersuchen

Verändern Sie die Kernelgröße des Gaussian Blur.

Erlaubt sind ungerade Größen, z.B. `(3,3)`, `(5,5)`, `(9,9)` oder `(15,15)`.

Welche Einstellung reduziert das Rauschen gut, ohne die Kanten zu stark zu verwischen?

> **Binder-/Notebook-Hinweis:** Die lokale Lambda-Funktion ermöglicht es, die Kernelgröße zu verändern und die Zelle erneut auszuführen, ohne neue globale Variablen zu deklarieren. Lokal wäre diese Konstruktion nicht notwendig.

In [ ]:
[]()
{
    // TODO: Kernelgröße verändern
    cv::Size kernel_size(15, 15);

    cv::Mat gaussian_test;

    cv::GaussianBlur(
        image_noisy,
        gaussian_test,
        kernel_size,
        0
    );

    std::cout << "Kernelgröße: "
              << kernel_size.width << " x "
              << kernel_size.height << std::endl;

    return show_image(gaussian_test);
}()

## 4. Kanten als lokale Änderungen

Eine Kante entsteht dort, wo sich die Bildintensität lokal stark verändert.

Der **Sobel-Operator** approximiert solche Änderungen getrennt in x- und y-Richtung. Aus beiden Komponenten kann anschließend die Stärke des lokalen Gradienten berechnet werden.

Für die Kantendetektion arbeiten wir zunächst mit einem Grauwertbild.

In [ ]:
cv::Mat gray;
cv::cvtColor(
    image,
    gray,
    cv::COLOR_BGR2GRAY
);

cv::Mat sobel_x;
cv::Mat sobel_y;

cv::Sobel(
    gray,
    sobel_x,
    CV_32F,
    1, 0,
    3
);

cv::Sobel(
    gray,
    sobel_y,
    CV_32F,
    0, 1,
    3
);

cv::Mat gradient_magnitude;
cv::magnitude(
    sobel_x,
    sobel_y,
    gradient_magnitude
);

cv::Mat gradient_vis;
cv::normalize(
    gradient_magnitude,
    gradient_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

> **Binder-/Notebook-Hinweis:** Grauwertbild und Gradientenbild werden in zwei Zellen ausgegeben. Lokal könnten beide gleichzeitig mit `cv::imshow()` dargestellt werden.

### Grauwertbild

In [ ]:
show_image(gray)

### Sobel – Gradientenstärke

In [ ]:
show_image(gradient_vis)

Die Kantenkarte enthält nicht mehr die ursprünglichen Farben oder Flächen. Stattdessen werden vor allem **Übergänge** hervorgehoben.

Damit können beispielsweise Konturen, Linien oder geometrische Strukturen für nachfolgende Verarbeitungsschritte vorbereitet werden.

## 5. Segmentierung: Welche Pixel gehören zusammen?

Bei der Segmentierung wird für jeden Pixel entschieden, zu welchem Bereich er gehört.

Für eine einfache binäre Segmentierung gilt beispielsweise:

$M(x,y)=
\begin{cases}
255, & \text{Pixel erfüllt die Bedingung}\\
0, & \text{Pixel erfüllt die Bedingung nicht}
\end{cases}$

Das Ergebnis ist eine **binäre Maske**.

Zur Einordnung:

| Aufgabe | Typisches Ergebnis |
|---|---|
| Klassifikation | Klasse |
| Objektdetektion | Klasse + Bounding Box |
| Segmentierung | Maske / Pixelzugehörigkeit |

Wir beginnen mit einer einfachen Segmentierung über Wertebereiche.

### Segmentierung im BGR-Raum

Der gesuchte breite Bereich ist grau. Für Grau sind die drei Farbkanäle ungefähr gleich groß.

Mit `cv::inRange()` können wir einen Minimal- und Maximalwert für jeden Kanal vorgeben. Pixel innerhalb dieses Bereichs erhalten in der Maske den Wert 255, alle anderen den Wert 0.

In [ ]:
cv::Scalar lower_bgr(100, 100, 100);
cv::Scalar upper_bgr(180, 180, 180);

cv::Mat mask_bgr;
cv::inRange(
    image,
    lower_bgr,
    upper_bgr,
    mask_bgr
);

cv::Mat result_bgr;
cv::bitwise_and(
    image,
    image,
    result_bgr,
    mask_bgr
);

> **Binder-/Notebook-Hinweis:** Ausgangsbild, Maske und Ergebnis werden getrennt dargestellt. Lokal könnten sie mit mehreren `cv::imshow()`-Fenstern gleichzeitig betrachtet werden.

### Ausgangsbild

In [ ]:
show_image(image)

### BGR-Maske

In [ ]:
show_image(mask_bgr)

### Segmentiertes Ergebnis

In [ ]:
show_image(result_bgr)

Die Segmentierung findet den breiten grauen Bereich in der Mitte und den grauen Schriftzug.

**Wichtig:** Der Threshold erkennt noch kein Objekt. Er entscheidet nur:

> Dieser Pixel liegt innerhalb meines Wertebereichs.

### Probieren Sie selbst

Verändern Sie `lower_bgr` und `upper_bgr`.

- Können Sie den grünen Bereich vollständig auswählen?

> **Binder-/Notebook-Hinweis:** Die Testwerte werden lokal innerhalb einer Lambda-Funktion definiert, damit die Zelle beim Experimentieren wiederholt ausgeführt werden kann. In einem lokalen C++-Programm wäre diese Kapselung nicht notwendig.

In [ ]:
[]()
{
    // TODO: Wertebereich verändern
    cv::Scalar lower_bgr_test(110, 110, 110);
    cv::Scalar upper_bgr_test(170, 170, 170);

    cv::Mat mask_bgr_test;

    cv::inRange(
        image,
        lower_bgr_test,
        upper_bgr_test,
        mask_bgr_test
    );

    return show_image(mask_bgr_test);
}()

## 6. Problem: Beleuchtung

RGB/BGR koppelt **Farbe und Helligkeit** direkt in den Kanalwerten.

Wir simulieren eine dunklere Beleuchtung, indem wir alle Pixelwerte reduzieren. Für uns bleibt der breite Bereich weiterhin grau, seine numerischen BGR-Werte ändern sich jedoch.

Wir verwenden anschließend **dieselben BGR-Grenzwerte wie zuvor**.

In [ ]:
cv::Mat image_dark;

image.convertTo(
    image_dark,
    -1,
    0.70,
    0
);

cv::Mat mask_bgr_dark;

cv::inRange(
    image_dark,
    lower_bgr,
    upper_bgr,
    mask_bgr_dark
);

cv::Mat result_bgr_dark;

cv::bitwise_and(
    image_dark,
    image_dark,
    result_bgr_dark,
    mask_bgr_dark
);

> **Binder-/Notebook-Hinweis:** Die Ergebnisse werden getrennt ausgegeben. Lokal könnten dunkleres Bild, Maske und Ergebnis parallel in mehreren `cv::imshow()`-Fenstern angezeigt werden.

### Dunkleres Bild

In [ ]:
show_image(image_dark)

### Gleicher BGR-Threshold

In [ ]:
show_image(mask_bgr_dark)

### Ergebnis

In [ ]:
show_image(result_bgr_dark)

Das Objekt hat sich geometrisch nicht verändert. Trotzdem liefert derselbe BGR-Threshold ein anderes Ergebnis.

### Segmentierung im HSV-Raum

Im HSV-Raum können Sättigung und Helligkeit getrennt betrachtet werden.

Für Grau ist der Hue-Wert kaum relevant. Entscheidend ist vor allem:

$S \approx 0$

Über \(V\) können wir zusätzlich zwischen dunklen, grauen und sehr hellen Bereichen unterscheiden.

Wir definieren deshalb einen Bereich mit geringer Sättigung und einer passenden Value-Spanne.

In [ ]:
cv::Mat hsv_dark;

cv::cvtColor(
    image_dark,
    hsv_dark,
    cv::COLOR_BGR2HSV
);

cv::Scalar lower_hsv(0, 0, 60);
cv::Scalar upper_hsv(179, 45, 150);

cv::Mat mask_hsv_dark;

cv::inRange(
    hsv_dark,
    lower_hsv,
    upper_hsv,
    mask_hsv_dark
);

cv::Mat result_hsv_dark;

cv::bitwise_and(
    image_dark,
    image_dark,
    result_hsv_dark,
    mask_hsv_dark
);

> **Binder-/Notebook-Hinweis:** Auch beim HSV-Vergleich werden die Bilder in einzelnen Zellen ausgegeben. Lokal könnten dieselben Ergebnisse mit mehreren `cv::imshow()`-Aufrufen gleichzeitig dargestellt werden.

### Dunkleres Bild

In [ ]:
show_image(image_dark)

### HSV-Maske

In [ ]:
show_image(mask_hsv_dark)

### HSV-Segmentierung

In [ ]:
show_image(result_hsv_dark)

HSV kann die Auswahl in manchen Situationen robuster machen, weil **Farbcharakteristik, Sättigung und Helligkeit getrennt beschrieben werden**.

Aber auch HSV erkennt noch kein Objekt: Die zweite graue Region erfüllt ebenfalls unseren Farbbereich.

### Mini-Aufgabe

Verändern Sie die HSV-Grenzen. Beobachten Sie insbesondere die Auswirkungen von

- maximaler Sättigung `S_max`,
- minimalem Value `V_min`,
- maximalem Value `V_max`.

> **Binder-/Notebook-Hinweis:** Die Grenzwerte liegen innerhalb einer Lambda-Funktion, damit die Zelle nach Änderungen erneut ausgeführt werden kann. Lokal wäre diese zusätzliche Kapselung nicht notwendig.

In [ ]:
[]()
{
    // TODO: HSV-Grenzen verändern
    cv::Scalar lower_hsv_test(0, 0, 60);
    cv::Scalar upper_hsv_test(179, 45, 150);

    cv::Mat mask_hsv_test;

    cv::inRange(
        hsv_dark,
        lower_hsv_test,
        upper_hsv_test,
        mask_hsv_test
    );

    return show_image(mask_hsv_test);
}()

## 7. Von Pixeln zu Regionen

Eine Maske beantwortet nur die Frage:

> Welche Pixel erfüllen meine Bedingung?

Zusammenhängende Gruppen von Pixeln können anschließend als **Konturen** extrahiert werden.

Mit `cv::findContours()` wechseln wir damit von einzelnen Pixeln zu zusammenhängenden Regionen. Für jede Region können anschließend geometrische Eigenschaften bestimmt werden, z.B.

- Fläche,
- Position,
- Breite und Höhe,
- Seitenverhältnis.

Damit können wir zusätzlich Wissen über unser gesuchtes Objekt verwenden.

In [ ]:
std::vector<std::vector<cv::Point>> contours;

cv::Mat mask_for_contours = mask_hsv_dark.clone();

cv::findContours(
    mask_for_contours,
    contours,
    cv::RETR_EXTERNAL,
    cv::CHAIN_APPROX_SIMPLE
);

cv::Mat contour_image = image_dark.clone();

for (const auto& contour : contours)
{
    cv::Rect rect = cv::boundingRect(contour);

    cv::rectangle(
        contour_image,
        rect,
        cv::Scalar(0, 255, 0),
        2
    );
}

std::cout << "Gefundene Regionen: "
          << contours.size()
          << std::endl;

> **Binder-/Notebook-Hinweis:** Das Ergebnis wird über `show_image(...)` im Browser dargestellt. Lokal würde man dafür beispielsweise `cv::imshow("Bounding Boxes", contour_image)` verwenden.

In [ ]:
show_image(contour_image)

### Welche Region ist das gesuchte Objekt?

Für unsere synthetische Szene wissen wir:

- der Zielbereich ist relativ groß,
- er ist horizontal,
- er ist deutlich breiter als hoch.

Diese Information können wir als einfache geometrische Regeln formulieren.

Das ist ein typisches Beispiel für **klassische Bildverarbeitung**: Wir kombinieren Pixelregeln mit bekanntem Wissen über die erwartete Geometrie.

In [ ]:
cv::Mat detected_image = image_dark.clone();

for (const auto& contour : contours)
{
    cv::Rect rect = cv::boundingRect(contour);

    double area = cv::contourArea(contour);

    double aspect_ratio =
        static_cast<double>(rect.width) /
        static_cast<double>(rect.height);

    // Einfache geometrische Regeln
    if (area > 1000.0 && aspect_ratio > 3.0)
    {
        cv::rectangle(
            detected_image,
            rect,
            cv::Scalar(0, 255, 0),
            4
        );

        std::cout << "Gefundene Zielregion:" << std::endl;
        std::cout << "  x, y = "
                  << rect.x << ", "
                  << rect.y << std::endl;

        std::cout << "  Breite, Höhe = "
                  << rect.width << ", "
                  << rect.height << std::endl;

        std::cout << "  Fläche = "
                  << std::fixed
                  << std::setprecision(1)
                  << area << std::endl;

        std::cout << "  Seitenverhältnis = "
                  << std::setprecision(2)
                  << aspect_ratio << std::endl;
    }
}

> **Binder-/Notebook-Hinweis:** Das markierte Ergebnis wird wieder mit der Notebook-Hilfsfunktion angezeigt. Lokal würde dafür normalerweise `cv::imshow()` verwendet.

In [ ]:
show_image(detected_image)

### Probieren Sie selbst

Verändern Sie die geometrischen Regeln.

- Was passiert, wenn die Mindestfläche zu klein gewählt wird?
- Welche Regeln wären problematisch, wenn das Objekt gedreht wäre?

> **Binder-/Notebook-Hinweis:** Die gesamte Übung wird in einer Lambda-Funktion ausgeführt. So können `min_area`, `min_aspect_ratio` und das Ergebnisbild beim erneuten Ausführen der Zelle neu angelegt werden. In einem normalen lokalen C++-Programm wäre diese Konstruktion nicht nötig.

In [ ]:
[]()
{
    // TODO: Eigene geometrische Regeln definieren
    double min_area = 5000.0;
    double min_aspect_ratio = 2.0;

    cv::Mat result_geometry = image_dark.clone();

    for (const auto& contour : contours)
    {
        cv::Rect rect = cv::boundingRect(contour);

        double area = cv::contourArea(contour);

        double aspect_ratio =
            static_cast<double>(rect.width) /
            static_cast<double>(rect.height);

        if (
            area > min_area &&
            aspect_ratio > min_aspect_ratio
        )
        {
            cv::rectangle(
                result_geometry,
                rect,
                cv::Scalar(0, 255, 0),
                3
            );
        }
    }

    return show_image(result_geometry);
}()

## 8. Grenzen klassischer Segmentierung

Unsere Pipeline funktioniert, weil wir bereits viel über das gesuchte Objekt wissen:

- ungefähre Farbe bzw. Helligkeit,
- Größe,
- Form,
- Seitenverhältnis.

Schwieriger wird die Methode bei

- wechselnder Beleuchtung,
- Schatten und Reflexionen,
- ähnlichen Farben im Hintergrund,
- komplexen Szenen,
- stark variierenden Objekten.

Der vollständige Ablauf in diesem Notebook war:

**Kamerabild → Filterung → Farbraum → Threshold → binäre Maske → Konturen → geometrische Regeln**

Klassische Bildverarbeitung funktioniert besonders gut, wenn wir **wissen, wonach wir suchen und unter welchen Bedingungen**.

## 9. Selbstcheck

Beantworten Sie die Fragen zunächst ohne in die Vorlesungsunterlagen zu schauen.

1. Warum kann die Nachbarschaft eines Pixels mehr Information enthalten als der einzelne Pixel selbst?
2. Was berechnet ein Kernel grundsätzlich an einer Bildposition?
3. Was ist der Unterschied zwischen Box Blur und Gaussian Blur?
4. Warum können zu große Glättungsfilter problematisch sein?
5. Was ist das Ergebnis einer einfachen binären Segmentierung?
6. Warum kann derselbe RGB/BGR-Threshold bei veränderter Beleuchtung scheitern?
7. Was ist der Vorteil des HSV-Farbraums für manche Segmentierungsaufgaben?
8. Warum ist eine segmentierte Maske noch keine Objekterkennung?
9. Welche zusätzlichen Informationen können aus Konturen gewonnen werden?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. Die lokale Umgebung enthält Struktur, z.B. gleichmäßige Regionen oder starke Änderungen an Kanten.
2. Eine gewichtete Kombination der Pixelwerte in einer lokalen Nachbarschaft.
3. Beim Box Blur werden Nachbarpixel gleich gewichtet; beim Gaussian Blur erhalten Pixel nahe dem Zentrum typischerweise ein größeres Gewicht.
4. Sie reduzieren zwar Rauschen, verwischen aber gleichzeitig Kanten und kleine Strukturen.
5. Eine Maske, in der Pixel typischerweise als passend bzw. nicht passend markiert werden.
6. Weil Farbe und Helligkeit direkt in den Kanalwerten gekoppelt sind und sich diese Werte mit der Beleuchtung ändern.
7. Hue, Saturation und Value können getrennt betrachtet werden.
8. Ein Threshold klassifiziert Pixel anhand ihrer Werte, kennt aber die semantische Bedeutung der zusammenhängenden Region nicht.
9. Beispielsweise Fläche, Bounding Box, Breite, Höhe und Seitenverhältnis.

</details>

## 10. Take-away

Für die Präsenz-LV sollten Sie folgende Punkte mitnehmen:

- Rohbilder enthalten neben relevanter Information auch Störungen.
- Kernels verarbeiten Pixel im Kontext ihrer lokalen Nachbarschaft.
- Glättungsfilter können Rauschen reduzieren, beeinflussen aber auch Kanten und Details.
- Schwellenwerte erzeugen aus Pixelwerten eine binäre Maske.
- Der gewählte Farbraum beeinflusst, wie einfach sich bestimmte Bereiche segmentieren lassen.
- Segmentierung klassifiziert zunächst Pixel – nicht automatisch Objekte.
- Konturen und geometrische Regeln können aus segmentierten Pixeln zusammenhängende Regionen ableiten.
- Klassische Bildverarbeitung ist besonders wirksam, wenn die erwarteten Eigenschaften und Umgebungsbedingungen gut bekannt sind.

### Ausblick

In der Vorlesung werden diese Grundlagen systematisch eingeordnet und auf weitere Methoden der klassischen Bildverarbeitung und Merkmalsextraktion vorbereitet.